# PUMP digital features

Adapted from `clean_extract_digital_features(3).ipynb`. The FAN numerical feature definitions are retained where applicable; PUMP loading, identities, states, and region selection replace the FAN-specific parts.

Run the cells in order. Put the two supplied PUMP CSVs beside the notebook, or edit `META_PATH` and `TIMESTAMPS_PATH`. The raw sensor files must exist at the paths in the metadata; use `PATH_REPLACEMENTS` if the drive/root changed. Dependencies: `numpy pandas polars scipy matplotlib tqdm` and `PyWavelets`.

- Default: healthy + single faults, determined from nonzero fault-code positions. The supplied data selects **315 paired samples / 18 classes**; `INCLUDE_MULTIFAULTS=True` selects all **613 pairs / 38 classes**.
- Use every interval between consecutive signalwise changepoints: **Air has regions 0–1; Water has regions 0–3**. Intervals are `[start, end)`. Supplied indices refer to raw-file rows; rows are never filtered or reordered before slicing.
- One output row per `fault_id@pump_id@sample_id`, with both states joined by ID. Example feature: `CV1_air_0_col_0_rms`. Region 0 means the first interval; it does not label an operating condition by itself.
- Required channels match the supplied PUMP reference: AC1/GYR have three axes; VRY has two columns; CV1, CTB, CTR, SMG, PRS and WTF use column 0. PRS/WTF apply only to Water. No extra channels are silently added.
- The uploaded metadata references **legacy data only**. Legacy `Timestamp` is seconds. A future merged file may use `firmware_timestamp` or `F_Timestamp` in microseconds. New electrical channel equivalents must be explicitly set in `NEW_RAW_COLUMNS`; the older inference mapping is not assumed to prove physical/calibration equivalence.
- Output files are saved in `pump_features`. All four notebooks are independent and use the same sample keys; no inference config, checkpoint or helper file is needed.

AC1 supplies accelerometer axes and the acceleration resultant; GYR supplies gyroscope axes. Each uses its own timestamps, sampling rate and changepoints. FAN's 2^14 window, 10–1000 Hz filter, FFT/CWT statistics, 75–125 / 175–225 Hz bands and band ratios are retained. These bands are inherited descriptors, not a claim that they are optimal for PUMP. Windows are centred within each region with 0.25-second edge margins. An unsupported filter/wavelet band raises an error instead of writing NaN.

In [1]:
import os

# Conservative thread limits to reduce sustained CPU load.
# Restart the kernel before running this notebook.
os.environ["POLARS_MAX_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import pandas as pd
import numpy as np
import polars as pl
import pywt

from tqdm import tqdm
from scipy.signal import find_peaks, butter, sosfiltfilt
from scipy.stats import skew, kurtosis
from scipy.fft import rfft, rfftfreq

In [2]:
from pathlib import Path

# Only edit these paths if the CSVs are not beside this notebook.
META_PATH = Path("pump_meta_combined_all_faults_mapped.csv")
TIMESTAMPS_PATH = Path("pump_signalwise_timestamp_combined.csv")
OUTPUT_DIR = Path("pump_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_MULTIFAULTS = False  # Same healthy + single-fault selection as FAN.
STATES = ("air", "water")
REGION_EDGE_SECONDS = 0.25

# Optional relocation of the raw files; leave empty if the CSV paths are valid.
PATH_REPLACEMENTS = {}  # e.g. {r"D:\VGuard Ahemedebad Data": r"E:\PumpData"}

# These are the legacy columns used by the supplied PUMP feature/config files.
SIGNAL_COLUMNS = {
    "AC1": ["0", "1", "2"], "GYR": ["0", "1", "2"],
    "CTB": ["0"], "CTR": ["0"], "CV1": ["0"], "VRY": ["0", "1"],
    "SMG": ["0"], "PRS": ["0"], "WTF": ["0"],
}
SIGNALS_BY_STATE = {
    "air": ["SMG", "AC1", "VRY", "CV1", "GYR", "CTB", "CTR"],
    "water": ["SMG", "AC1", "VRY", "CV1", "GYR", "CTB", "CTR", "PRS", "WTF"],
}

# The supplied combined metadata contains legacy files only. For future raw
# analog.csv files, explicitly confirm electrical channel equivalents here.
# No old/new calibration or electrical equivalence is inferred from filenames.
NEW_RAW_COLUMNS = {
    "AC1": {"0": "ac1", "1": "ac2", "2": "ac3"},
    "GYR": {"0": "gy1", "1": "gy2", "2": "gy3"},
    "SMG": {"0": "mag"}, "PRS": {"0": "prs"}, "WTF": {"0": "wtf"},
    "CV1": {}, "VRY": {}, "CTB": {}, "CTR": {},
}

meta = pd.read_csv(META_PATH, dtype={"fault_id": str, "pump_id": str, "sample_id": str})
orig_ts = pd.read_csv(TIMESTAMPS_PATH, dtype=str)
meta["state"] = meta["state"].str.strip().str.lower()
orig_ts["state"] = orig_ts["state"].str.strip().str.lower()
for col in ["fault_id", "pump_id", "sample_id"]:
    meta[col] = meta[col].str.strip()
meta["sample_id_key"] = meta[["fault_id", "pump_id", "sample_id"]].agg("@".join, axis=1)
meta["pump_id_key"] = meta[["fault_id", "pump_id"]].agg("@".join, axis=1)
meta["state_id_key"] = meta["sample_id_key"] + "@" + meta["state"]
active_faults = meta["fault_id"].str.rstrip("_").map(lambda x: sum(c != "0" for c in x))
meta["fault_type"] = np.where(active_faults == 0, "nofault", np.where(active_faults == 1, "single", "multi"))
if not INCLUDE_MULTIFAULTS:
    meta = meta[meta["fault_type"] != "multi"].copy()
meta = meta.reset_index(drop=True)
if meta.empty or meta["state_id_key"].duplicated().any():
    raise ValueError("Metadata is empty or has duplicate sample/state keys.")
if not set(meta["state"]).issubset(STATES):
    raise ValueError("Unrecognized state in PUMP metadata.")
if orig_ts["id"].duplicated().any():
    raise ValueError("Duplicate signalwise changepoint IDs.")

orig_ts_lookup = {}
for row in orig_ts.itertuples(index=False):
    if not row.id.startswith(row.key + "_") or not row.id.endswith("@" + row.state):
        raise ValueError(f"Inconsistent signal/state in timestamp ID: {row.id}")
    cp = np.array(row.indices.split("#"), dtype=np.int64)
    if len(cp) < 2 or cp[0] < 0 or np.any(np.diff(cp) <= 0):
        raise ValueError(f"Invalid changepoints: {row.id}")
    orig_ts_lookup[row.id] = cp

# Pair Air/Water by explicit IDs, never by row position.
for sample_id, group in meta.groupby("sample_id_key", sort=False):
    if set(group["state"]) != set(STATES):
        raise ValueError(f"Missing Air/Water state: {sample_id}")
region_counts = {}
for index, row in meta.iterrows():
    for sig in SIGNALS_BY_STATE[row["state"]]:
        if sig not in meta or pd.isna(row[sig]) or not str(row[sig]).strip():
            raise ValueError(f"Missing {sig} path for {row['state_id_key']}")
        cp_key = f"{sig}_{row['state_id_key']}"
        if cp_key not in orig_ts_lookup:
            raise ValueError(f"Missing changepoints: {cp_key}")
        count = len(orig_ts_lookup[cp_key]) - 1
        previous = region_counts.setdefault(row["state"], count)
        if previous != count:
            raise ValueError(f"Inconsistent number of regions: {cp_key}")

print(f"{meta['sample_id_key'].nunique()} paired samples; {meta['fault_id'].nunique()} classes")
print("Regions per state:", region_counts)
meta.head()

315 paired samples; 18 classes
Regions per state: {'air': 2, 'water': 4}


,fault_id,pump_id,sample_id,state,AC1,GYR,CTB,CTR,CV1,VRY,PRS,SMG,WTF,sample_id_key,pump_id_key,state_id_key,fault_type
0,00000000000_,1,1,air,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,NaN,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,NaN,00000000000_@1@1,00000000000_@1,00000000000_@1@1@air,nofault
1,00000000000_,1,1,water,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,D:\VGuard Ahemedebad Data\Primary_20230727-31\...,00000000000_@1@1,00000000000_@1,00000000000_@1@1@water,nofault
2,00000000000_,1,4,air,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,00000000000_@1@4,00000000000_@1,00000000000_@1@4@air,nofault
3,00000000000_,1,4,water,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,00000000000_@1@4,00000000000_@1,00000000000_@1@4@water,nofault
4,00000000000_,2,1,air,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,D:\VGuard Ahemedebad Data\Primary_20230726\No ...,NaN,00000000000_@2@1,00000000000_@2,00000000000_@2@1@air,nofault


In [3]:
def resolve_path(value):
    path = str(value)
    for old_root, new_root in PATH_REPLACEMENTS.items():
        if path.startswith(old_root):
            path = str(new_root) + path[len(old_root):]
            break
    if os.name != "nt":
        path = path.replace("\\", "/")
    return path


def read_pump_signal(index, sig):
    """Read one signal without removing/reordering rows referenced by changepoints."""
    path = resolve_path(meta.loc[index, sig])
    header = pd.read_csv(path, nrows=0).columns.tolist()
    new_time = next((c for c in ("firmware_timestamp", "F_Timestamp") if c in header), None)
    timecol = new_time or "Timestamp"
    if timecol not in header:
        raise ValueError(f"Timestamp column missing: {path}")
    mapping = {}
    for col in SIGNAL_COLUMNS[sig]:
        # Prefer canonical signal names in merged files. Bare column_0/0
        # names apply only to separate legacy signal files.
        candidates = [f"{sig}_column_{col}"]
        if not new_time:
            candidates += [f"column_{col}", col]
        else:
            configured = NEW_RAW_COLUMNS.get(sig, {}).get(col)
            if configured:
                candidates.append(configured)
        found = next((c for c in candidates if c in header), None)
        if found is None:
            raise ValueError(f"Missing/unmapped {sig} column {col}: {path}. "
                             "For new raw electrical files, set NEW_RAW_COLUMNS explicitly.")
        mapping[found] = col
    orig = pl.read_csv(path, columns=[timecol] + list(mapping), n_threads=1,
                       low_memory=True, rechunk=False).rename({timecol: "Timestamp", **mapping})
    if new_time:
        orig = orig.with_columns((pl.col("Timestamp") / 1e6).alias("Timestamp"))
    times = orig["Timestamp"].to_numpy()
    valid_positions = np.flatnonzero(np.isfinite(times) & (times >= 0))
    if len(valid_positions) < 2:
        raise ValueError(f"Not enough valid timestamps: {path}")
    first, last = valid_positions[[0, -1]]
    duration = times[last] - times[first]
    if duration <= 0:
        raise ValueError(f"Invalid timestamp duration: {path}")
    fs = (last - first) / duration  # Legacy Timestamp is seconds; firmware time is microseconds.
    cp_key = f"{sig}_{meta.loc[index, 'state_id_key']}"
    cp = orig_ts_lookup[cp_key]
    if cp[-1] > orig.height:
        raise ValueError(f"Changepoints exceed raw-file row count: {cp_key}")
    return orig, fs, cp


def region_window(cp, region, fs, dist):
    """Fixed-size central window, with a margin inside both changepoints."""
    start, end = map(int, cp[region:region + 2])
    edge = int(np.ceil(REGION_EDGE_SECONDS * fs))
    available = end - start - 2 * edge
    if available < dist:
        raise ValueError(f"Region {region}: needs {dist} samples plus edge margins; "
                         f"only {available} usable samples. No crossing into another region.")
    ll = start + edge + (available - dist) // 2
    return ll, ll + dist


def finite_window(values, context):
    x = np.asarray(values, dtype=np.float64)
    if len(x) == 0 or not np.all(np.isfinite(x)):
        raise ValueError(f"Empty/non-finite feature window: {context}")
    return x


def feature_prefix(index, sig, region, col):
    return f"{sig}_{meta.loc[index, 'state']}_{region}_col_{col}"


def collect_features(extractor, description):
    """One output row per fault/pump/sample; state and region live in column names."""
    rows = []
    for sample_id, group in tqdm(meta.groupby("sample_id_key", sort=False),
                                 total=meta["sample_id_key"].nunique(), desc=description):
        row = {}
        for index in group.index:
            package = extractor(index)
            overlap = set(row).intersection(package)
            if overlap:
                raise ValueError(f"Duplicate state/region features: {overlap}")
            row.update(package)
        identity = group.iloc[0]
        for key in ["sample_id_key", "fault_id", "pump_id", "sample_id", "pump_id_key", "fault_type"]:
            row[key] = identity[key]
        rows.append(row)
        gc.collect()
    result = pd.DataFrame(rows)
    if result.empty or result.isna().any().any():
        raise ValueError("Empty/incomplete feature table; inspect sample regions and signals.")
    numeric = result.select_dtypes(include=np.number)
    if not np.isfinite(numeric.to_numpy()).all():
        raise ValueError("Non-finite feature values; inspect the raw signals.")
    return result

In [10]:
def bandpower(lo, hi, freqs, mag):
    mask = (freqs >= lo) & (freqs < hi)
    _fft = mag[mask]
    _fft_freq = freqs[mask]
    if len(_fft) == 0:
        return [0.0, 0.0, 0.0, 0.0, 0.0]
    peak_indices, _ = find_peaks(_fft, height=np.max(_fft) * 0.1)
    if len(peak_indices) == 0:
        top_idx = np.argmax(_fft)
        top_peak_freqs = _fft_freq[top_idx]
        top_peak_mags = _fft[top_idx]
    else:
        peak_freqs = _fft_freq[peak_indices]
        peak_mags = _fft[peak_indices]
        top_peaks_idx = np.argmax(peak_mags)
        top_peak_freqs = peak_freqs[top_peaks_idx]
        top_peak_mags = peak_mags[top_peaks_idx]
    band_power = float(np.sum(_fft ** 2))
    total_power = float(np.sum(mag ** 2)) + 1e-12
    relative_band_power = band_power / total_power
    peak_to_median = float(top_peak_mags / (np.median(_fft) + 1e-12))
    return [float(top_peak_freqs), float(top_peak_mags), band_power, float(relative_band_power), peak_to_median]

def compute_freq_features(key: str, x: np.ndarray, fs: float) -> dict:
    N = len(x)
    freqs = rfftfreq(N, 1 / fs)[:N // 2]
    mag = np.abs(rfft(x))[:N // 2]
    max_mag = np.max(mag)
    if max_mag > 1e-12:
        mag = mag / max_mag
    else:
        mag = np.zeros_like(mag)
    mean_, std_ = (mag.mean(), mag.std())
    skew_, kurt = (skew(mag), kurtosis(mag))
    p = mag / (mag.sum() + 1e-12)
    entropy = -np.sum(p * np.log2(p + 1e-12))
    peak_idx = mag.argmax()
    peak_freq = freqs[peak_idx]
    centroid = np.sum(freqs * mag) / (mag.sum() + 1e-12)
    return (freqs, mag, {f'{key}_fft_mean': float(mean_), f'{key}_fft_std': float(std_), f'{key}_fft_skew': float(skew_), f'{key}_fft_kurtosis': float(kurt), f'{key}_fft_entropy': float(entropy), f'{key}_spectral_centroid': float(centroid)})

def compute_cwt_features(key: str, x: np.ndarray, fs: float, wavelet: str='cmor1.5-1.0', scales: np.ndarray=np.arange(1, 128)) -> tuple:
    coeffs, freqs = pywt.cwt(x, scales, wavelet, sampling_period=1 / fs, method= "fft")
    power = np.abs(coeffs)
    power **= 2
    del coeffs
    max_power = float(power.max())
    raw_power_sum = float(power.sum())
    if max_power <= 1e-12:
        power.fill(0.0)
    else:
        power /= max_power
    total = power.sum()
    p = power / (total + 1e-12)
    entropy = -np.sum(p * np.log2(p + 1e-12))
    del p
    feats = {f'{key}_wavelet_mean_energy': float(power.mean()), f'{key}_wavelet_std_energy': float(power.std()), f'{key}_wavelet_max_energy': float(max_power), f'{key}_wavelet_max_energy_ratio': float(max_power / (raw_power_sum + 1e-12)), f'{key}_wavelet_entropy': float(entropy)}
    return (freqs, power, feats)

def temporal_modulation_features(power: np.ndarray, key='') -> dict:
    e = power if power.ndim == 1 else power.mean(axis=0)
    threshold = e.mean() + e.std()
    peaks, _ = find_peaks(e, height=threshold)
    intervals = np.diff(peaks) if len(peaks) > 1 else np.array([])
    return {f'{key}_num_spikes': float(len(peaks)), f'{key}_mean_spike_interval': float(intervals.mean() if intervals.size else 0.0), f'{key}_std_spike_interval': float(intervals.std() if intervals.size else 0.0), f'{key}_spike_energy_ratio': float(e[peaks].sum() / (e.sum() + 1e-12))}

def tf_contrast_features(key, power: np.ndarray) -> dict:
    row = power.mean(axis=1)
    col = power.mean(axis=0)
    return {f'{key}_contrast_freq': float(row.std() / (row.mean() + 1e-12)), f'{key}_contrast_time': float(col.std() / (col.mean() + 1e-12))}


def get_features(key, x, orig_sample_freq, xlim, bands):
    x = finite_window(x, key)
    if not 0 < xlim[0] < xlim[1] < orig_sample_freq / 2:
        raise ValueError(f"{key}: {xlim} band is invalid for sampling rate {orig_sample_freq}")
    sos = butter(5, xlim, 'bp', fs=orig_sample_freq, output='sos')
    x = sosfiltfilt(sos, x)
    if np.std(x) <= 1e-12:
        raise ValueError(f'{key}: stable-region signal is nearly constant')
    feats = {}
    rfft_freqs, mag, freq_feats = compute_freq_features(key, x, orig_sample_freq)
    feats.update(freq_feats)
    # freqs, power, cwt_feats = compute_cwt_features(key, x, orig_sample_freq)
    wavelet = "cmor1.5-1.0"
    min_freq = min(lo for lo, hi in bands)

    max_scale = max(
        127,
        int(np.ceil(
            pywt.central_frequency(wavelet) * orig_sample_freq / min_freq
        ))
    )

    scales = np.arange(1, max_scale + 1)

    freqs, power, cwt_feats = compute_cwt_features(
        key, x, orig_sample_freq,
        wavelet=wavelet,
        scales=scales
    )
    feats.update(cwt_feats)
    feats.update(tf_contrast_features(key, power))
    band_ranks = {}
    band_power_values = {}
    for lo, hi in bands:
        band_mask = (freqs >= lo) & (freqs <= hi)
        if not np.any(band_mask):
            raise ValueError(f"{key}: no CWT scales cover band {lo}–{hi} Hz at fs={orig_sample_freq}")
        band_pow = power[band_mask, :]
        spike_feats = temporal_modulation_features(band_pow, key=f'{key}_{lo}_{hi}')
        feats.update(spike_feats)
        package = bandpower(lo, hi, rfft_freqs, mag)
        feats[f'{key}_band_peak_freq_{lo}_{hi}'] = float(package[0])
        feats[f'{key}_band_peak_mag_{lo}_{hi}'] = float(package[1])
        feats[f'{key}_band_power_{lo}_{hi}'] = float(package[2])
        feats[f'{key}_relative_band_power_{lo}_{hi}'] = float(package[3])
        feats[f'{key}_band_peak_to_median_{lo}_{hi}'] = float(package[4])
        band_power_values[f'{lo}_{hi}'] = package[3]
        band_ranks[f'{lo}_{hi}'] = package[1]
    if len(bands) >= 2:
        band1 = f'{bands[0][0]}_{bands[0][1]}'
        band2 = f'{bands[1][0]}_{bands[1][1]}'
        feats[f'{key}_log_band_power_ratio'] = float(np.log((band_power_values[band1] + 1e-12) / (band_power_values[band2] + 1e-12)))
    band_ranks = dict(sorted(band_ranks.items(), reverse=True, key=lambda x: x[1]))
    for i, k in enumerate(band_ranks):
        feats[f'{key}_rank_{k}'] = i
    return feats

In [11]:
DIGITAL_DIST = 2**14
DIGITAL_BANDS = [(75, 125), (175, 225)]
DIGITAL_FILTER = (10, 1000)

def extract_all_acc_features(index=0, resultant=True):
    feats = {}
    for sig in ["AC1", "GYR"]:
        orig, fs, cp = read_pump_signal(index, sig)
        for region in range(len(cp) - 1):
            ll, ul = region_window(cp, region, fs, DIGITAL_DIST)
            axes = []
            for col in SIGNAL_COLUMNS[sig]:
                x = finite_window(orig[col].slice(ll, ul - ll).to_numpy(), f"{sig}/{col}")
                axes.append(x)
                key = feature_prefix(index, sig, region, col)
                feats.update(get_features(key, x, fs, DIGITAL_FILTER, DIGITAL_BANDS))
            if resultant and sig == "AC1":
                x = np.sqrt(np.sum(np.stack(axes)**2, axis=0))
                key = feature_prefix(index, sig, region, "resultant")
                feats.update(get_features(key, x, fs, DIGITAL_FILTER, DIGITAL_BANDS))
        del orig
    return feats

In [12]:
feats_pd = collect_features(extract_all_acc_features, "PUMP digital")
feats_pd.head()

PUMP digital: 100%|██████████| 315/315 [1:00:44<00:00, 11.57s/it]


,AC1_air_0_col_0_fft_mean,AC1_air_0_col_0_fft_std,AC1_air_0_col_0_fft_skew,AC1_air_0_col_0_fft_kurtosis,AC1_air_0_col_0_fft_entropy,AC1_air_0_col_0_spectral_centroid,AC1_air_0_col_0_wavelet_mean_energy,AC1_air_0_col_0_wavelet_std_energy,AC1_air_0_col_0_wavelet_max_energy,AC1_air_0_col_0_wavelet_max_energy_ratio,...,GYR_water_3_col_2_band_peak_to_median_175_225,GYR_water_3_col_2_log_band_power_ratio,GYR_water_3_col_2_rank_75_125,GYR_water_3_col_2_rank_175_225,sample_id_key,fault_id,pump_id,sample_id,pump_id_key,fault_type
0,0.004581,0.026278,15.388335,382.435395,9.172139,721.607912,0.067680,0.089512,233470.562165,0.000003,...,2.973681,1.396725,0,1,00000000000_@1@1,00000000000_,1,1,00000000000_@1,nofault
1,0.002432,0.015562,35.900017,2098.152777,9.354360,614.144891,0.216349,0.285913,141127.998448,0.000001,...,1.329683,1.191176,0,1,00000000000_@1@4,00000000000_,1,4,00000000000_@1,nofault
2,0.005549,0.025239,14.625204,401.669531,9.634872,718.235468,0.156814,0.202840,367221.665902,0.000001,...,1.544329,1.471705,0,1,00000000000_@2@1,00000000000_,2,1,00000000000_@2,nofault
3,0.003893,0.019313,23.066793,977.237936,9.511762,684.639508,0.198492,0.252297,414487.606044,0.000001,...,1.812219,1.090191,0,1,00000000000_@2@2,00000000000_,2,2,00000000000_@2,nofault
4,0.006950,0.031186,12.265148,260.086722,9.561303,772.488204,0.096557,0.119125,312386.492834,0.000002,...,2.548993,1.022958,0,1,00000000000_@2@3,00000000000_,2,3,00000000000_@2,nofault


In [13]:
feats_pd.to_csv(OUTPUT_DIR / "PUMP_digital_feats_updt.csv", index=False)
print("Saved", feats_pd.shape)

Saved (315, 1434)
